# Projet de groupe : codes de démarrage des neuf groupes

*Introduction au Machine Learning, CNFSDP 2026-2027, Rois Céti DIMBAMBA*

Ce notebook rassemble les codes de préparation du **guide du projet** (section « Fiches de démarrage ») : pour chaque groupe, il lit le fichier de l'EDS Congo 2011-2012, construit la population et la cible, calcule la **proportion pondérée** de la cible, et prépare les bases prêtes à l'emploi :

- `X`, `y`, `w` : variables explicatives, cible et poids, pour le volet supervisé ;
- `Z`, `illus` : variables actives et variables illustratives (cible, poids), pour le volet non supervisé.

Une cellule de **vérification** suit chaque groupe : elle compare l'effectif et le taux pondéré obtenus aux valeurs annoncées dans le guide. Le tableau final récapitule les neuf groupes.

**Avant de lancer.**
1. Décompresser les quatre archives (`CGHR61DT.zip`, `CGIR61DT.zip`, `CGKR61DT.zip`, `CGPR61DT.zip`) dans un dossier `donnees/`, à côté de ce notebook (sous-dossiers acceptés). Le code trouve seul les fichiers `.DTA`.
2. Lancer la cellule du bloc commun, puis les cellules d'un groupe (chaque groupe est indépendant des autres).

**Rappel.** Les données DHS ne sont jamais déposées sur GitHub ni sur aucun dossier partagé. Ce notebook ne contient aucune donnée : seulement du code.

**Les poids.** `hv005`/$10^6$ pour les ménages (fichier HR), `v005`/$10^6$ pour les femmes (IR) et les enfants (KR, poids de la mère). Pour une part de population à partir du fichier des ménages : `hv005`/$10^6$ × `hv012` (membres du ménage). Proportion pondérée : $\hat p=\sum_i w_iy_i/\sum_i w_i$, en Python `np.average(y, weights=w)`.

## Bloc commun

In [ ]:
# Bloc commun à tous les groupes : à exécuter une fois, avant le code du groupe.
import os, re, glob
import numpy as np
import pandas as pd

DOSSIER = "donnees"   # dossier où les 4 archives .zip ont été décompressées

def chemin_dhs(fichier):
    """Chemin du fichier Stata du fichier 'HR', 'IR', 'KR' ou 'PR'."""
    motif = re.compile(rf"^CG{fichier}61\w*\.DTA$", re.IGNORECASE)
    trouves = [f for f in glob.glob(os.path.join(DOSSIER, "**", "*"), recursive=True)
               if motif.match(os.path.basename(f))]
    if not trouves:
        raise FileNotFoundError(f"Pas de CG{fichier}61...DTA dans « {DOSSIER} »")
    return trouves[0]

def lire_dhs(fichier, colonnes):
    """Lit les colonnes demandées, en gardant les codes numériques."""
    chemin = chemin_dhs(fichier)
    with pd.io.stata.StataReader(chemin) as lecteur:
        disponibles = set(lecteur.variable_labels())
    absentes = [c for c in colonnes if c not in disponibles]
    if absentes:
        print("Absentes du fichier, ignorées :", absentes)
    df = pd.read_stata(chemin, columns=[c for c in colonnes if c in disponibles],
                       convert_categoricals=False)
    print(f"{os.path.basename(chemin)} : {df.shape[0]} lignes, {df.shape[1]} colonnes")
    return df

def manquant(serie, codes):
    """Remplace par NaN les codes spéciaux (ne sait pas, manquant, hors limites...)."""
    return serie.where(~serie.isin(codes))

def indicatrice(serie, codes, manquants=(99,)):
    """1 si le code est dans la liste, 0 sinon, NaN si le code est manquant."""
    x = serie.isin(codes).astype(float)
    return x.where(serie.notna() & ~serie.isin(manquants))

def taux_pondere(y, poids):
    """Proportion pondérée, en % : somme(poids * y) / somme(poids)."""
    return 100 * np.average(y, weights=poids)

def base_supervisee(df, cible, numeriques, categorielles):
    """X (une indicatrice par modalité des catégorielles), cible y, poids w."""
    num = [c for c in numeriques if c in df]
    cat = [c for c in categorielles if c in df]
    X = df[num + cat].copy()
    for c in cat:   # une modalité « manquant » plutôt que de perdre la ligne
        X[c] = X[c].astype("Int64").astype(str).replace("<NA>", "manquant")
    X = pd.get_dummies(X, columns=cat, dtype=int)
    return X, df[cible].astype(int), df["poids"]

def base_non_supervisee(df, actives, illustratives):
    """Variables actives (sans valeur manquante) et variables illustratives."""
    act = [c for c in actives if c in df]
    garde = df[act].notna().all(axis=1)
    print(f"Non supervisé : {garde.sum()} lignes complètes sur {len(df)}")
    return df.loc[garde, act], df.loc[garde, illustratives]

def resume(X, y, w):
    print("X :", X.shape, "| cible :", round(100 * y.mean(), 1), "% (non pondéré),",
          round(taux_pondere(y, w), 1), "% (pondéré)")
    manq = X.isna().sum()
    print("Valeurs manquantes à imputer :", dict(manq[manq > 0]))

In [ ]:
# Vérification : effectif et taux pondéré comparés aux valeurs du guide
verifications = []
def verifier(groupe, libelle, obtenu, attendu, n=None, n_attendu=None, tolerance=0.3):
    ok_taux = abs(obtenu - attendu) <= tolerance
    ok_n = (n is None) or (n == n_attendu)
    verifications.append({"groupe": groupe, "indicateur": libelle, "obtenu (%)": round(obtenu, 1),
                          "attendu (%)": attendu, "effectif": n, "effectif attendu": n_attendu,
                          "conforme": "oui" if ok_taux and ok_n else "NON"})
    print(f"{libelle} : {obtenu:.1f} % (attendu {attendu} %)",
          "" if n is None else f"; effectif {n} (attendu {n_attendu})",
          "-> conforme" if ok_taux and ok_n else "-> À VÉRIFIER")

## Groupe 1 : Richesse : indice national

Fichier `HR` ; cible : ménages pauvres : `hv270` ∈ {1, 2}. Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 1 - Richesse : indice national (fichier HR, un ménage par ligne)
biens = ["hv206", "hv207", "hv208", "hv209", "hv210", "hv211", "hv212", "hv221",
         "hv243a", "hv243b", "hv244", "hv246", "hv247"]
logement = ["hv201", "hv205", "hv213", "hv214", "hv215", "hv216", "hv226"]
menage = ["hhid", "hv005", "hv009", "hv010", "hv011", "hv012", "hv014",
          "hv024", "hv025", "hv219", "hv220", "hv270", "hv271"]
hr = lire_dhs("HR", menage + biens + logement)

# Poids et cible
hr["poids"] = hr["hv005"] / 1e6                   # poids de sondage du ménage
hr["pauvre"] = hr["hv270"].isin([1, 2]).astype(int)   # 2 quintiles les plus bas
print("Ménages pauvres :", round(taux_pondere(hr["pauvre"], hr["poids"]), 1), "%")
print("Population pauvre :",
      round(taux_pondere(hr["pauvre"], hr["poids"] * hr["hv012"]), 1), "%")

# Instruction du chef de ménage (fichier PR : hv101 = 1 pour le chef)
pr = lire_dhs("PR", ["hhid", "hv101", "hv106"])
chef = pr[pr["hv101"] == 1].drop_duplicates("hhid")[["hhid", "hv106"]]
hr = hr.merge(chef.rename(columns={"hv106": "instr_chef"}), on="hhid", how="left")
hr["instr_chef"] = manquant(hr["instr_chef"], [8, 9])

# Codes spéciaux : âge du chef 98-99 (ne sait pas, manquant)
hr["hv220"] = manquant(hr["hv220"], [98, 99])

# Variables actives du non supervisé : 1 = possède / logement de qualité
biens = [b for b in biens if b in hr]
for b in biens:
    hr[b] = manquant(hr[b], [9])                  # 0 = non, 1 = oui, 9 = manquant
hr["eau_amelioree"] = indicatrice(hr["hv201"], [11, 12, 13, 14, 21, 31, 41, 51, 71])
hr["wc_ameliore"] = indicatrice(hr["hv205"], [11, 12, 13, 15, 21, 22, 41])
for v, nom in [("hv213", "sol_fini"), ("hv214", "mur_fini"), ("hv215", "toit_fini")]:
    hr[nom] = indicatrice(hr[v], range(31, 40))   # codes 31 à 39 : matériau fini
hr["cuisine_propre"] = indicatrice(hr["hv226"], [1, 2, 3, 4])  # électricité, gaz
hr["pers_par_piece"] = hr["hv009"] / manquant(hr["hv216"], [0, 99])

# Base du supervisé : caractéristiques démographiques (ni biens ni logement)
X, y, w = base_supervisee(hr, "pauvre",
    numeriques=["hv009", "hv010", "hv011", "hv014", "hv220"],
    categorielles=["hv024", "hv025", "hv219", "instr_chef"])
resume(X, y, w)

# Base du non supervisé : biens et logement ; hv270, hv271 et la cible en illustratif
actives = biens + ["eau_amelioree", "wc_ameliore", "sol_fini", "mur_fini",
                   "toit_fini", "cuisine_propre", "pers_par_piece"]
Z, illus = base_non_supervisee(hr, actives, ["pauvre", "hv270", "hv271", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(1, "ménages pauvres (poids hv005)", taux_pondere(hr["pauvre"], hr["poids"]), 40.8,
         len(hr), 11632)
verifier(1, "population pauvre (poids hv005 x hv012)",
         taux_pondere(hr["pauvre"], hr["poids"] * hr["hv012"]), 39.7)
print("Sans pondération :", round(100 * hr["pauvre"].mean(), 1), "% des ménages")

## Groupe 2 : Richesse : indices urbain et rural

Fichier `HR` ; cible : ménages pauvres (`hv270` ∈ {1, 2}), urbain et rural séparés (`hv025`). Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 2 - Richesse : indices urbain et rural (fichier HR, un ménage par ligne)
biens = ["hv206", "hv207", "hv208", "hv209", "hv210", "hv211", "hv212", "hv221",
         "hv243a", "hv243b", "hv244", "hv246", "hv247"]
logement = ["hv201", "hv205", "hv213", "hv214", "hv215", "hv216", "hv226"]
menage = ["hhid", "hv005", "hv009", "hv010", "hv011", "hv012", "hv014",
          "hv024", "hv025", "hv219", "hv220", "hv270", "hv271"]
hr = lire_dhs("HR", menage + biens + logement)

# Poids, cible et milieu (hv025 : 1 = urbain, 2 = rural)
hr["poids"] = hr["hv005"] / 1e6
hr["pauvre"] = hr["hv270"].isin([1, 2]).astype(int)
hr["milieu"] = hr["hv025"].map({1: "urbain", 2: "rural"})
for m, d in hr.groupby("milieu"):
    print(f"{m} : {len(d)} ménages ; pauvres",
          round(taux_pondere(d["pauvre"], d["poids"]), 1), "% des ménages,",
          round(taux_pondere(d["pauvre"], d["poids"] * d["hv012"]), 1),
          "% de la population")

# Instruction du chef de ménage (fichier PR : hv101 = 1 pour le chef)
pr = lire_dhs("PR", ["hhid", "hv101", "hv106"])
chef = pr[pr["hv101"] == 1].drop_duplicates("hhid")[["hhid", "hv106"]]
hr = hr.merge(chef.rename(columns={"hv106": "instr_chef"}), on="hhid", how="left")
hr["instr_chef"] = manquant(hr["instr_chef"], [8, 9])
hr["hv220"] = manquant(hr["hv220"], [98, 99])

# Variables actives (mêmes définitions que le groupe 1)
biens = [b for b in biens if b in hr]
for b in biens:
    hr[b] = manquant(hr[b], [9])
hr["eau_amelioree"] = indicatrice(hr["hv201"], [11, 12, 13, 14, 21, 31, 41, 51, 71])
hr["wc_ameliore"] = indicatrice(hr["hv205"], [11, 12, 13, 15, 21, 22, 41])
for v, nom in [("hv213", "sol_fini"), ("hv214", "mur_fini"), ("hv215", "toit_fini")]:
    hr[nom] = indicatrice(hr[v], range(31, 40))
hr["cuisine_propre"] = indicatrice(hr["hv226"], [1, 2, 3, 4])
hr["pers_par_piece"] = hr["hv009"] / manquant(hr["hv216"], [0, 99])
actives = biens + ["eau_amelioree", "wc_ameliore", "sol_fini", "mur_fini",
                   "toit_fini", "cuisine_propre", "pers_par_piece"]

# Une base par milieu : supervisé (sans hv025) et non supervisé
bases = {}
for m in ["urbain", "rural"]:
    d = hr[hr["milieu"] == m]
    X, y, w = base_supervisee(d, "pauvre",
        numeriques=["hv009", "hv010", "hv011", "hv014", "hv220"],
        categorielles=["hv024", "hv219", "instr_chef"])
    Z, illus = base_non_supervisee(d, actives, ["pauvre", "hv270", "hv271", "poids"])
    bases[m] = {"X": X, "y": y, "w": w, "Z": Z, "illus": illus}
    print(m, end=" : "); resume(X, y, w)
# Modèle unique, pour comparer : tous les ménages, avec le milieu hv025 parmi les X
X_tous, y_tous, w_tous = base_supervisee(hr, "pauvre",
    numeriques=["hv009", "hv010", "hv011", "hv014", "hv220"],
    categorielles=["hv024", "hv025", "hv219", "instr_chef"])
display(bases["urbain"]["X"].head(), bases["rural"]["Z"].describe().T.round(2))

In [ ]:
for m, attendu, n_att in [("urbain", 13.9, 3150), ("rural", 86.5, 8482)]:
    d = hr[hr["milieu"] == m]
    verifier(2, f"population {m} pauvre (poids hv005 x hv012)",
             taux_pondere(d["pauvre"], d["poids"] * d["hv012"]), attendu, len(d), n_att)

## Groupe 3 : Vaccination complète

Fichier `KR` ; cible : les 8 vaccins de base reçus (`h2` à `h9` ∈ {1, 2, 3}), enfants vivants de 12 à 23 mois. Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 3 - Vaccination complète (fichier KR, un enfant né depuis 5 ans par ligne)
vaccins = ["h2", "h3", "h4", "h5", "h6", "h7", "h8", "h9"]
enfant = ["caseid", "midx", "v005", "v008", "b3", "b4", "b5", "bord"]
mere = ["v012", "v106", "v190", "v025", "v024", "v130", "v136", "v151",
        "v157", "v158", "v159", "m14", "m15"]
kr = lire_dhs("KR", enfant + mere + vaccins)

# Population : enfants vivants (b5 = 1) de 12 à 23 mois ; âge = v008 - b3 (en mois)
kr["age_mois"] = kr["v008"] - kr["b3"]
df = kr[(kr["b5"] == 1) & kr["age_mois"].between(12, 23)].copy()
df["poids"] = df["v005"] / 1e6           # poids de la mère, porté par l'enfant

# Cible : 8 vaccins reçus (1 = date sur carte, 2 = déclaré, 3 = marqué sur carte)
recu = df[vaccins].isin([1, 2, 3])
df["vacc_complete"] = recu.all(axis=1).astype(int)
print(len(df), "enfants de 12-23 mois ; vaccination complète :",
      round(taux_pondere(df["vacc_complete"], df["poids"]), 1), "%")
print("Couverture par vaccin (%) :",
      {v: round(taux_pondere(recu[v], df["poids"]), 1) for v in vaccins})

# Codes spéciaux et recodages
for v in ["v157", "v158", "v159"]:
    df[v] = manquant(df[v], [9])                  # 0 jamais ... 3 presque chaque jour
df["m14"] = manquant(df["m14"], [98, 99])  # visites prénatales (dernière naissance)
df["accouch_etab"] = indicatrice(df["m15"], range(20, 50))   # né en établissement
df["v130"] = manquant(df["v130"], [99])
df["urbain"] = (df["v025"] == 1).astype(int)

# Base du supervisé ; à écarter : h0 à h10, h1 (carte), et tout autre vaccin
X, y, w = base_supervisee(df, "vacc_complete",
    numeriques=["age_mois", "bord", "v012", "v136", "m14", "v157", "v158", "v159",
                "accouch_etab"],
    categorielles=["b4", "v106", "v190", "v025", "v024", "v130", "v151"])
resume(X, y, w)

# Base du non supervisé : profil de la mère et du ménage ; la cible en illustratif
actives = ["v012", "v106", "v190", "urbain", "v136", "bord", "v157", "v158", "v159",
           "m14", "accouch_etab"]
Z, illus = base_non_supervisee(df, actives, ["vacc_complete", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(3, "vaccination complète (poids v005)", taux_pondere(df["vacc_complete"], df["poids"]),
         38.1, len(df), 1884)
print("Sans pondération :", round(100 * df["vacc_complete"].mean(), 1), "%")

## Groupe 4 : Abandon entre DTC1 et DTC3

Fichier `KR` ; cible : DTC1 reçu (`h3` ∈ {1, 2, 3}) mais pas le DTC3 (`h7` ∉ {1, 2, 3}), enfants de 12 à 23 mois. Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 4 - Abandon entre DTC1 et DTC3 (fichier KR, un enfant par ligne)
vaccins = ["h2", "h3", "h4", "h5", "h6", "h7", "h8", "h9"]
enfant = ["caseid", "midx", "v005", "v008", "b3", "b4", "b5", "bord"]
mere = ["v012", "v106", "v190", "v025", "v024", "v130", "v136", "v151",
        "v157", "v158", "v159", "m14", "m15"]
kr = lire_dhs("KR", enfant + mere + vaccins)

# Enfants vivants de 12 à 23 mois
kr["age_mois"] = kr["v008"] - kr["b3"]
base = kr[(kr["b5"] == 1) & kr["age_mois"].between(12, 23)].copy()
base["poids"] = base["v005"] / 1e6
base["dtc1"] = base["h3"].isin([1, 2, 3]).astype(int)
base["dtc3"] = base["h7"].isin([1, 2, 3]).astype(int)
print("12-23 mois :", len(base),
      "; DTC1 :", round(taux_pondere(base["dtc1"], base["poids"]), 1),
      "% ; DTC3 :", round(taux_pondere(base["dtc3"], base["poids"]), 1), "%")

# Population : ceux qui ont reçu le DTC1 ; cible : DTC3 non reçu (abandon)
df = base[base["dtc1"] == 1].copy()
df["abandon"] = 1 - df["dtc3"]
print(len(df), "enfants ayant reçu le DTC1 ; abandon :",
      round(taux_pondere(df["abandon"], df["poids"]), 1), "%")

# Codes spéciaux et recodages
for v in ["v157", "v158", "v159"]:
    df[v] = manquant(df[v], [9])
df["m14"] = manquant(df["m14"], [98, 99])
df["accouch_etab"] = indicatrice(df["m15"], range(20, 50))
df["v130"] = manquant(df["v130"], [99])
df["urbain"] = (df["v025"] == 1).astype(int)

# Base du supervisé ; à écarter : h0 à h10 (dont h3, h5, h7), h1 (carte)
X, y, w = base_supervisee(df, "abandon",
    numeriques=["age_mois", "bord", "v012", "v136", "m14", "v157", "v158", "v159",
                "accouch_etab"],
    categorielles=["b4", "v106", "v190", "v025", "v024", "v130", "v151"])
resume(X, y, w)

# Base du non supervisé ; l'abandon en illustratif
actives = ["v012", "v106", "v190", "urbain", "v136", "bord", "v157", "v158", "v159",
           "m14", "accouch_etab"]
Z, illus = base_non_supervisee(df, actives, ["abandon", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(4, "DTC1, 12-23 mois (poids v005)", taux_pondere(base["dtc1"], base["poids"]), 74.5,
         len(base), 1884)
verifier(4, "DTC3, 12-23 mois (poids v005)", taux_pondere(base["dtc3"], base["poids"]), 56.3)
verifier(4, "abandon parmi les DTC1 (poids v005)", taux_pondere(df["abandon"], df["poids"]), 25.0,
         len(df), 1290)

## Groupe 5 : Vaccination contre la rougeole

Fichier `KR` ; cible : rougeole reçue (`h9` ∈ {1, 2, 3}), enfants vivants de 12 à 23 mois. Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 5 - Vaccination contre la rougeole (fichier KR, un enfant par ligne)
enfant = ["caseid", "midx", "v005", "v008", "b3", "b4", "b5", "bord", "h9"]
mere = ["v012", "v106", "v190", "v025", "v024", "v130", "v136", "v151",
        "v157", "v158", "v159", "m14", "m15"]
kr = lire_dhs("KR", enfant + mere)

# Population : enfants vivants de 12 à 23 mois
kr["age_mois"] = kr["v008"] - kr["b3"]
df = kr[(kr["b5"] == 1) & kr["age_mois"].between(12, 23)].copy()
df["poids"] = df["v005"] / 1e6

# Cible : rougeole reçue (h9 = 1, 2 ou 3) ; 0 = non, 8 = ne sait pas comptent comme non
df["rougeole"] = df["h9"].isin([1, 2, 3]).astype(int)
print(len(df), "enfants de 12-23 mois ; rougeole :",
      round(taux_pondere(df["rougeole"], df["poids"]), 1), "%")

# Codes spéciaux et recodages
for v in ["v157", "v158", "v159"]:
    df[v] = manquant(df[v], [9])
df["m14"] = manquant(df["m14"], [98, 99])
df["accouch_etab"] = indicatrice(df["m15"], range(20, 50))
df["v130"] = manquant(df["v130"], [99])
df["urbain"] = (df["v025"] == 1).astype(int)

# Base du supervisé ; l'âge compte (vaccin vers 9 mois) ; à écarter : h0 à h10
X, y, w = base_supervisee(df, "rougeole",
    numeriques=["age_mois", "bord", "v012", "v136", "m14", "v157", "v158", "v159",
                "accouch_etab"],
    categorielles=["b4", "v106", "v190", "v025", "v024", "v130", "v151"])
resume(X, y, w)

# Base du non supervisé (ACP) ; la cible en illustratif
actives = ["v012", "v106", "v190", "urbain", "v136", "bord", "v157", "v158", "v159",
           "m14", "accouch_etab"]
Z, illus = base_non_supervisee(df, actives, ["rougeole", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(5, "rougeole (poids v005)", taux_pondere(df["rougeole"], df["poids"]), 74.9, len(df), 1884)

## Groupe 6 : Retard de croissance

Fichier `KR` ; cible : `hw70` < -200, enfants vivants mesurés (`hw70` < 9996). Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 6 - Retard de croissance (fichier KR, un enfant par ligne)
enfant = ["caseid", "midx", "v005", "b0", "b4", "b5", "bord", "b11", "m18",
          "h11", "hw1", "hw70"]
mere = ["v012", "v106", "v190", "v025", "v024", "v136", "v113", "v116",
        "v438", "v445"]
kr = lire_dhs("KR", enfant + mere)

# Population : enfants vivants mesurés ; hw70 >= 9996 = hors limites ou manquant
df = kr[(kr["b5"] == 1) & (kr["hw70"] < 9996)].copy()
df["poids"] = df["v005"] / 1e6

# Cible : score taille-pour-âge (x 100) inférieur à -2 écarts types
df["retard"] = (df["hw70"] < -200).astype(int)
print(len(df), "enfants mesurés ; retard de croissance :",
      round(taux_pondere(df["retard"], df["poids"]), 1), "%")

# Codes spéciaux et recodages
df["taille_mere"] = manquant(df["v438"], range(9990, 10000)) / 10    # en cm
df["imc_mere"] = manquant(df["v445"], range(9990, 10000)) / 100      # en kg/m2
df["m18"] = manquant(df["m18"], [8, 9])   # taille à la naissance : 1 très gros ... 5
df["premier_ne"] = df["b11"].isna().astype(int)  # pas d'intervalle pour un premier-né
df["jumeau"] = (df["b0"] > 0).astype(int)
df["diarrhee"] = indicatrice(df["h11"], [1, 2], manquants=(8, 9))
df["eau_amelioree"] = indicatrice(df["v113"], [11, 12, 13, 14, 21, 31, 41, 51, 71])
df["wc_ameliore"] = indicatrice(df["v116"], [11, 12, 13, 15, 21, 22, 41])
df["urbain"] = (df["v025"] == 1).astype(int)

# Base du supervisé ; à écarter : hw70, hw71, hw72 (indices), hw2, hw3 (mesures)
X, y, w = base_supervisee(df, "retard",
    numeriques=["hw1", "bord", "b11", "m18", "v012", "v136", "taille_mere", "imc_mere",
                "premier_ne", "jumeau", "diarrhee", "eau_amelioree", "wc_ameliore"],
    categorielles=["b4", "v106", "v190", "v025", "v024"])
resume(X, y, w)

# Base du non supervisé : enfant, mère, ménage ; la cible en illustratif
actives = ["hw1", "bord", "m18", "taille_mere", "imc_mere", "v106", "v190", "urbain",
           "v136", "eau_amelioree", "wc_ameliore"]
Z, illus = base_non_supervisee(df, actives, ["retard", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(6, "retard de croissance (poids v005)", taux_pondere(df["retard"], df["poids"]), 23.1,
         len(df), 4475)

## Groupe 7 : Insuffisance pondérale

Fichier `KR` ; cible : `hw71` < -200, enfants vivants mesurés (`hw71` < 9996). Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 7 - Insuffisance pondérale (fichier KR, un enfant par ligne)
enfant = ["caseid", "midx", "v005", "b0", "b4", "b5", "bord", "b11", "m18",
          "h11", "hw1", "hw71"]
mere = ["v012", "v106", "v190", "v025", "v024", "v136", "v113", "v116",
        "v438", "v445"]
kr = lire_dhs("KR", enfant + mere)

# Population : enfants vivants mesurés ; hw71 >= 9996 = hors limites ou manquant
df = kr[(kr["b5"] == 1) & (kr["hw71"] < 9996)].copy()
df["poids"] = df["v005"] / 1e6

# Cible : score poids-pour-âge (x 100) inférieur à -2 écarts types
df["insuff_pond"] = (df["hw71"] < -200).astype(int)
print(len(df), "enfants mesurés ; insuffisance pondérale :",
      round(taux_pondere(df["insuff_pond"], df["poids"]), 1), "%")

# Codes spéciaux et recodages
df["taille_mere"] = manquant(df["v438"], range(9990, 10000)) / 10    # en cm
df["imc_mere"] = manquant(df["v445"], range(9990, 10000)) / 100      # en kg/m2
df["m18"] = manquant(df["m18"], [8, 9])   # taille à la naissance : 1 très gros ... 5
df["premier_ne"] = df["b11"].isna().astype(int)  # pas d'intervalle pour un premier-né
df["jumeau"] = (df["b0"] > 0).astype(int)
df["diarrhee"] = indicatrice(df["h11"], [1, 2], manquants=(8, 9))
df["eau_amelioree"] = indicatrice(df["v113"], [11, 12, 13, 14, 21, 31, 41, 51, 71])
df["wc_ameliore"] = indicatrice(df["v116"], [11, 12, 13, 15, 21, 22, 41])
df["urbain"] = (df["v025"] == 1).astype(int)

# Base du supervisé ; à écarter : hw71, hw70, hw72 (indices), hw2, hw3 (mesures)
X, y, w = base_supervisee(df, "insuff_pond",
    numeriques=["hw1", "bord", "b11", "m18", "v012", "v136", "taille_mere", "imc_mere",
                "premier_ne", "jumeau", "diarrhee", "eau_amelioree", "wc_ameliore"],
    categorielles=["b4", "v106", "v190", "v025", "v024"])
resume(X, y, w)

# Base du non supervisé : enfant, mère, ménage ; la cible en illustratif
actives = ["hw1", "bord", "m18", "taille_mere", "imc_mere", "v106", "v190", "urbain",
           "v136", "eau_amelioree", "wc_ameliore"]
Z, illus = base_non_supervisee(df, actives, ["insuff_pond", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(7, "insuffisance pondérale (poids v005)", taux_pondere(df["insuff_pond"], df["poids"]),
         11.1, len(df), 4475)

## Groupe 8 : Contraception moderne

Fichier `IR` ; cible : méthode moderne (`v313` = 3), femmes en union (`v502` = 1). Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 8 - Contraception moderne (fichier IR, une femme de 15 à 49 ans par ligne)
femme = ["caseid", "v005", "v502", "v501", "v313", "v012", "v106", "v190", "v025",
         "v024", "v130", "v136", "v201", "v218", "v157", "v158", "v159", "v714",
         "v511", "v743a", "v701", "v505"]
ir = lire_dhs("IR", femme)

# Population : femmes actuellement en union (v502 = 1 : mariées ou en concubinage)
df = ir[ir["v502"] == 1].copy()
df["poids"] = df["v005"] / 1e6                    # poids de sondage de la femme

# Cible : méthode moderne utilisée actuellement (v313 = 3)
df["moderne"] = (df["v313"] == 3).astype(int)
print(len(df), "femmes en union ; contraception moderne :",
      round(taux_pondere(df["moderne"], df["poids"]), 1), "%")

# Codes spéciaux et recodages
for v in ["v157", "v158", "v159"]:
    df[v] = manquant(df[v], [9])
df["v714"] = manquant(df["v714"], [9])            # travaille actuellement : 0 / 1
df["v701"] = manquant(df["v701"], [8, 9])         # instruction du conjoint : 0 à 3
df["v130"] = manquant(df["v130"], [99])
df["polygamie"] = indicatrice(df["v505"], range(1, 98), manquants=(98, 99))
df["decide_sante"] = indicatrice(df["v743a"], [1, 2, 3], manquants=(9,))
df["urbain"] = (df["v025"] == 1).astype(int)

# Base du supervisé ; à écarter : v312, v361 à v364 (méthode, usage passé, prévu),
# v626a (besoins non satisfaits), v602 et v605 (contiennent « stérilisée »)
X, y, w = base_supervisee(df, "moderne",
    numeriques=["v012", "v136", "v201", "v218", "v511", "v157", "v158", "v159",
                "v714", "polygamie", "decide_sante"],
    categorielles=["v106", "v190", "v025", "v024", "v130", "v501", "v701"])
resume(X, y, w)

# Base du non supervisé : profil des femmes ; la cible en illustratif
actives = ["v012", "v106", "v190", "urbain", "v201", "v218", "v511", "v157", "v158",
           "v159", "v714", "v701", "polygamie", "decide_sante"]
Z, illus = base_non_supervisee(df, actives, ["moderne", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(8, "contraception moderne (poids v005)", taux_pondere(df["moderne"], df["poids"]), 20.0,
         len(df), 6750)

## Groupe 9 : Besoins non satisfaits en contraception

Fichier `IR` ; cible : besoin non satisfait (`v626a` ∈ {1, 2}), femmes en union (`v502` = 1). Détails dans la fiche du groupe (guide du projet).

In [ ]:
# Groupe 9 - Besoins non satisfaits en contraception (fichier IR, une femme par ligne)
femme = ["caseid", "v005", "v502", "v501", "v626a", "v012", "v106", "v190", "v025",
         "v024", "v130", "v136", "v201", "v218", "v157", "v158", "v159", "v714",
         "v511", "v743a", "v701", "v505"]
ir = lire_dhs("IR", femme)

# Population : femmes actuellement en union (v502 = 1 : mariées ou en concubinage)
df = ir[ir["v502"] == 1].copy()
df["poids"] = df["v005"] / 1e6                    # poids de sondage de la femme

# Cible : besoin non satisfait pour espacer (v626a = 1) ou pour limiter (v626a = 2)
df["besoin_ns"] = df["v626a"].isin([1, 2]).astype(int)
print(len(df), "femmes en union ; besoins non satisfaits :",
      round(taux_pondere(df["besoin_ns"], df["poids"]), 1), "%")

# Codes spéciaux et recodages
for v in ["v157", "v158", "v159"]:
    df[v] = manquant(df[v], [9])
df["v714"] = manquant(df["v714"], [9])            # travaille actuellement : 0 / 1
df["v701"] = manquant(df["v701"], [8, 9])         # instruction du conjoint : 0 à 3
df["v130"] = manquant(df["v130"], [99])
df["polygamie"] = indicatrice(df["v505"], range(1, 98), manquants=(98, 99))
df["decide_sante"] = indicatrice(df["v743a"], [1, 2, 3], manquants=(9,))
df["urbain"] = (df["v025"] == 1).astype(int)

# Base du supervisé ; à écarter : ce qui construit v626a (v312, v313, v361 à v364,
# v602, v605, grossesse v213, v225, aménorrhée v405, abstinence v406)
X, y, w = base_supervisee(df, "besoin_ns",
    numeriques=["v012", "v136", "v201", "v218", "v511", "v157", "v158", "v159",
                "v714", "polygamie", "decide_sante"],
    categorielles=["v106", "v190", "v025", "v024", "v130", "v501", "v701"])
resume(X, y, w)

# Base du non supervisé : profil des femmes ; la cible en illustratif
actives = ["v012", "v106", "v190", "urbain", "v201", "v218", "v511", "v157", "v158",
           "v159", "v714", "v701", "polygamie", "decide_sante"]
Z, illus = base_non_supervisee(df, actives, ["besoin_ns", "poids"])
display(X.head(), Z.describe().T.round(2))

In [ ]:
verifier(9, "besoins non satisfaits (poids v005)", taux_pondere(df["besoin_ns"], df["poids"]), 18.4,
         len(df), 6750)

## Récapitulatif des vérifications

Un écart de plus de 0,3 point, ou un effectif différent, signale une erreur dans la population ou dans la cible (filtre oublié, code spécial non traité, mauvais poids) : relire la fiche du groupe et le dictionnaire (`.MAP`).

In [ ]:
display(pd.DataFrame(verifications).astype({"effectif": "Int64", "effectif attendu": "Int64"}))

## Volet non supervisé (commun à tous les groupes)

À lancer après les cellules d'un groupe : il utilise les tableaux `Z` (variables actives) et `illus` (cible et poids) de ce groupe. Pour le groupe 2, choisir d'abord un milieu : `Z, illus = bases["urbain"]["Z"], bases["urbain"]["illus"]`. Ajuster `q` (nombre d'axes) d'après l'éboulis et `k` (nombre de classes) d'après le dendrogramme.

In [ ]:
# Volet non supervisé, commun à tous les groupes : à lancer après le code du groupe
# (il utilise Z, les variables actives, et illus : la cible et le poids).
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

cible = illus.columns[0]                     # variable illustrative : la cible
Zs = StandardScaler().fit_transform(Z)       # ACP normée : centrer, réduire

# 1. ACP : part d'inertie de chaque axe (éboulis), on garde les premiers axes
acp = PCA().fit(Zs)
print("Inertie expliquée (%) :", (100 * acp.explained_variance_ratio_).round(1))
q = 5                                  # nombre d'axes gardés, à choisir sur l'éboulis
coord = acp.transform(Zs)[:, :q]

# 2. Lire les axes : corrélation de chaque variable active avec les 3 premiers axes
cor = pd.DataFrame(acp.components_[:3].T * np.sqrt(acp.explained_variance_[:3]),
                   index=Z.columns, columns=["F1", "F2", "F3"])
display(cor.round(2).sort_values("F1"))

# 3. Premier plan factoriel, individus colorés selon la cible (illustrative)
plt.scatter(coord[:, 0], coord[:, 1], c=illus[cible], cmap="coolwarm", s=4, alpha=0.4)
plt.xlabel("F1"); plt.ylabel("F2"); plt.title(f"Premier plan, couleur : {cible}")
plt.show()

# 4. CAH de Ward sur les q premiers axes (sur 5 000 individus au plus, pour la mémoire)
rng = np.random.default_rng(0)
ech = rng.choice(len(coord), size=min(5000, len(coord)), replace=False)
arbre = linkage(coord[ech], method="ward")
dendrogram(arbre, truncate_mode="lastp", p=30); plt.title("Dendrogramme"); plt.show()
k = 4                                        # nombre de classes, lu sur le dendrogramme
classes_cah = fcluster(arbre, t=k, criterion="maxclust")

# 5. Consolidation : k-means partant des centres des classes de la CAH, sur tous
centres = np.vstack([coord[ech][classes_cah == c].mean(axis=0)
                     for c in range(1, k + 1)])
km = KMeans(n_clusters=k, init=centres, n_init=1).fit(coord)
classe = pd.Series(km.labels_ + 1, index=Z.index, name="classe")

# 6. Décrire les classes : effectif, moyennes des actives, taux pondéré de la cible
profil = Z.groupby(classe).mean().round(2)
profil.insert(0, "effectif", classe.value_counts().sort_index())
profil.insert(1, f"{cible} (%)", illus.groupby(classe).apply(
    lambda d: round(taux_pondere(d[cible], d["poids"]), 1)))
display(profil)